# RootCause-SLM v2: block-level explanations

Everything runs here, nothing needed on your computer. The repo is kept in your Google Drive, so if the annotation stops (daily limit), you can come back the next day and run the cells again: it continues where it stopped. Set the runtime to T4 first (Runtime > Change runtime type).

1. annotate the 409 training and 123 test blocks with the teacher (API key needed, about 20-30 min)
2. fine-tune Qwen2.5-1.5B on them (about 15-30 min)
3. evaluate rules, teacher, v1, base model and v2 against the raw log (about 15 min)
4. download the results

In [ ]:
# The repo is cloned into your Google Drive, so the annotation progress survives
# a disconnection or a new session the next day.
from google.colab import drive
drive.mount("/content/drive")
import os
REPO = "/content/drive/MyDrive/RootCause-SLM"
if not os.path.exists(REPO):
    !git clone -q https://github.com/NafiyTP/RootCause-SLM.git "$REPO"
else:
    !cd "$REPO" && git pull -q
%cd $REPO
!pip install -q "transformers==4.46.3" peft accelerate requests

## 1. Annotation

Paste your API key when asked (it is not saved in the notebook). Groq by default; for Gemini, change `PROVIDER` to `"gemini"`.
If the cell stops (rate limit, disconnection), run it again: blocks already done are skipped.

About 530 calls of ~1,000 tokens each. If the free tier has a daily limit, the script stops cleanly and says so: run the cell again the next day, or switch to the other provider (the progress is kept per provider, so finish with the one you started).

**To continue later or with a new key:** run the first cell (Drive + repo), then the key cell, then the annotation cell again.

In [ ]:
import os, getpass
PROVIDER = "groq"   # or "gemini"
MODEL = ""          # empty = default (Groq: openai/gpt-oss-120b, Gemini: gemini-2.0-flash)
key_name = {"groq": "GROQ_API_KEY", "gemini": "GEMINI_API_KEY"}[PROVIDER]
# asks every time you run this cell. Several keys of the same account: separate them with commas,
# they are used in turn when one hits a limit or runs out of credit
os.environ[key_name] = getpass.getpass(f"{key_name} (key1,key2,...): ")
MODEL_ARG = f"--model {MODEL}" if MODEL else ""

In [ ]:
%cd /content/drive/MyDrive/RootCause-SLM
!python src/annotate_v2.py --split test --provider $PROVIDER $MODEL_ARG

In [ ]:
%cd /content/drive/MyDrive/RootCause-SLM
!python src/annotate_v2.py --split train --provider $PROVIDER $MODEL_ARG

Quick look at what the teacher wrote:

In [ ]:
%cd /content/drive/MyDrive/RootCause-SLM
import json
d = json.load(open('data/v2/test_annotated.json'))
for x in d[:5]:
    print(x['label'], x['category'], '->', x['teacher']['anomalous'], '|', x['teacher']['cause'])

## 2. Fine-tuning

If an import fails, restart the session (Runtime > Restart session), run the first cell again, then this one.

In [ ]:
%cd /content/drive/MyDrive/RootCause-SLM
!python src/train_v2.py

## 3. Evaluation

The v1 adapter (`modele_hdfs/`) is already in the repo.

In [ ]:
%cd /content/drive/MyDrive/RootCause-SLM
!python src/eval_v2.py

In [ ]:
%cd /content/drive/MyDrive/RootCause-SLM
from IPython.display import Markdown
Markdown(open('results/v2/metrics.md', encoding='utf-8').read())

## 4. Download

The zip has the annotations, the adapter and the results. Send me `results/v2/` and I update the README, the paper and the site.

In [ ]:
%cd /content/drive/MyDrive/RootCause-SLM
!zip -qr v2_results.zip data/v2 results/v2 modele_hdfs_v2
from google.colab import files
files.download('v2_results.zip')